# Forecasting Short-Term Stock Market Volatility

**Research question:** Can machine learning predict the volatility of a stock over its next five trading days using historical price movements, trading volume, and market trends?

## Introduction

I plan to build a regression model that analyzes stock price history and predicts the magnitude of future price fluctuations. 

I will focus of three things:

- **Prediction:** Estimate future volatility using data available at fixed time

- **Comparison:** Compare such ML algorithm with simple historical models.

**Note:** Such comparison is not trivial, see [Forecasting realized volatility: Does anything beat linear models?](https://www.sciencedirect.com/science/article/abs/pii/S0927539824000598?via%3Dihub)

- **Evaluation** Measure how accurately models predict volatility on unseen data.

### Import

In [1]:
# Imports and global settings. Everything the notebook needs lives here.
from pathlib import Path
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

# House plotting theme.
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["savefig.dpi"] = 200          # sharp figures for the PDF report
plt.rcParams["savefig.bbox"] = "tight"

# Paths.
DATA_DIR = Path("us_daily_data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Universe: 20 large caps across sectors, plus SPY as the market proxy.
TICKERS = [
    "AAPL", "MSFT", "NVDA", "AMD", "INTC",   # technology
    "JPM", "BAC", "V", "MA",                 # finance
    "WMT", "COST", "KO", "PG",               # consumer
    "JNJ", "PFE",                            # healthcare
    "XOM", "CVX", "CAT",                     # energy / industrial
    "DIS", "NFLX",                           # media
]
MARKET = "SPY"

# Experiment design.
HORIZON = 5                  # predict volatility over the next 5 trading days
START = "2018-01-01"         # first modelling date (load from 2017 for feature warm-up)
TRAIN_END = "2023-12-31"     # train: 2018-2023
VAL_END = "2024-12-31"       # validation: 2024, test: 2025 onward

# Reproducibility.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Imports ready. Seed fixed at", RANDOM_STATE)

Imports ready. Seed fixed at 42


## Data Collection

**Dataset used:** [Stooq Historical Data](https://stooq.com/db/h/?).

**Download data for usage:** U.S. → Daily → ASCII

Extract zip into the folder with this file. Expected workspace:

| Path | Type | Description |
| :--- | :--- | :--- |
| `root/` | Directory | Main project workspace |
| ├── `stock-market-volatility-predictor.ipynb` | Notebook | Core code, analysis, and volatility prediction model |
| └── `data/` | Directory | Raw historical market data files |

#### Expected bulk ASCII columns

```python3
<TICKER>,<PER>,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<VOL>,<OPENINT>
AAPL.US,D,20261001,000000,195.0,199.0,193.0,197.0,45000000,0
```

Run python code below to format downloaded file into us-daily-data folder.

In [4]:
# One-time data setup: flatten the extracted Stooq download (data/daily/us) into us-daily-data/.
import shutil
from pathlib import Path

src = Path("data") / "daily" / "us"      # what the Stooq zip creates when extracted here
dst = Path("us-daily-data")              # what the rest of the notebook reads
REQUIRED = ["nasdaq stocks", "nyse stocks", "nyse etfs"]

if not dst.exists():
    assert Path("data").is_dir(), (
        "Expected a 'data' folder next to this notebook. "
        "Download U.S. -> Daily -> ASCII from Stooq and extract the zip into this folder."
    )
    assert src.is_dir(), (
        f"Found 'data' but not '{src}'. Make sure you downloaded the U.S. Daily ASCII archive."
    )
    shutil.move(str(src), str(dst))
    # remove the now-empty data/daily and data folders (left alone if anything else is in them)
    for leftover in (Path("data") / "daily", Path("data")):
        if leftover.is_dir() and not any(leftover.iterdir()):
            leftover.rmdir()
    print(f"Moved {src} -> {dst}")
else:
    print(f"{dst} already exists, skipping the move.")

missing = [g for g in REQUIRED if not (dst / g).is_dir()]
assert not missing, f"{dst} is missing expected folders: {missing}"
assert any(dst.rglob("aapl.us.txt")), f"{dst} has no aapl.us.txt; the download looks incomplete."
print(f"Data ready: {sum(1 for _ in dst.rglob('*.txt')):,} files in {dst}")

us-daily-data already exists, skipping the move.
Data ready: 13,345 files in us-daily-data


### Dataset Structure (`us-daily-data/` — 13,345 Total Files)

| Directory / Subfolder | File Count | Alphabetical Range | Key Tickers Included |
| :--- | :---: | :--- | :--- |
| **`nasdaq stocks/`** | **4,642** | *3 subfolders* | |
| ↳ `1/` | 2,000 | `aaap` … `hviiu` | `AAPL`, `AMD`, `COST` |
| ↳ `2/` | 2,000 | `hvmc` … `sy` | `MSFT`, `NVDA`, `INTC`, `NFLX` |
| ↳ `3/` | 642 | `sybt` … `zyn` | `WMT` |
| **`nasdaq etfs/`** | **987** | `aadr` … `ztwo` *(no subfolders)* | `QQQ` |
| **`nyse stocks/`** | **4,488** | *3 subfolders* | |
| ↳ `1/` | 2,000 | `a` … `idvz` | `BAC`, `CAT`, `CVX`, `DIS` |
| ↳ `2/` | 2,000 | `idyn` … `tsm` | `JPM`, `MA`, `KO`, `PG`, `JNJ`, `PFE` |
| ↳ `3/` | 488 | `t_a` … `zws` | `V`, `XOM` |
| **`nyse etfs/`** | **2,905** | *2 subfolders* | |
| ↳ `1/` | 2,000 | `aaa` … `ptlc` | — |
| ↳ `2/` | 905 | `ptmc` … `ztax` | `SPY` |
| **`nysemkt stocks/`** | **321** | `accs` … `zone` *(no subfolders)* | — |
| **`nysemkt etfs/`** | **2** | `iqei`, `iqel` *(no subfolders)* | — |

* **`nasdaq stocks`** — Individual companies listed on Nasdaq. Tech-heavy: `AAPL`, `MSFT`, `NVDA`, `AMD`, `INTC`, `COST`, `NFLX`.
* **`nasdaq etfs`** — Funds listed on Nasdaq (`QQQ`, the Nasdaq-100 fund).
* **`nyse stocks`** — Individual companies listed on the New York Stock Exchange. Older blue chips: `JPM`, `KO`, `XOM`, `JNJ`.
* **`nyse etfs`** — Funds listed on NYSE (in practice its Arca platform, where most US ETFs trade).
* **`nysemkt stocks`** — Companies on NYSE American, formerly the American Stock Exchange (AMEX). Mostly small-cap companies.
* **`nysemkt etfs`** — The few funds listed on NYSE American.

##### Quick Term Guide

* **Stock vs. ETF** — A **stock** is a share of ownership in one company. An **ETF** (Exchange-Traded Fund) holds a basket of securities and trades like a single stock; `SPY` holds the S&P 500, so its price tracks the broader US market.
* **`SPY` (SPDR S&P 500 ETF Trust)** — An exchange-traded fund tracking the S&P 500 (`us-daily-data/nyse etfs/2/spy.us.txt`), used as a proxy feature to capture broad, market-wide volatility that individual stock histories miss.
* **Nasdaq vs. NYSE** — The two largest US stock exchanges. **Nasdaq** is an all-electronic exchange known for growth and tech companies; **NYSE** is the traditional floor-and-electronic exchange known for established corporations.
* **Blue Chips** — Large, well-established, financially sound companies with a long history of reliable performance (e.g., `JPM`, `KO`, `JNJ`).
* **S&P 500 / Nasdaq-100** — Market indexes. The **S&P 500** tracks 500 leading US companies across all sectors; the **Nasdaq-100** tracks the 100 largest non-financial companies on Nasdaq.
* **Small-Cap** — Companies with a smaller total market value (typically $300M–$2B), often carrying higher growth potential and higher volatility.
* **Numbered Subfolders (`1`, `2`, `3`)** — Arbitrary chunks with no market meaning. **Stooq** (the historical market data provider) splits large folders alphabetically into batches of ~2,000 files (`nasdaq stocks/1` starts at `aaap` and holds 2,000 files).
* **Unified Schema** — The CSV file format is identical across all six folders, so the exact same loading code works everywhere.

## Data Cleaning

As seen from above downloaded data set is vast, so we will first make some data cleaning. We will keep the following stocks:

| Sector | Tickers | Count |
| :--- | :--- | :---: |
| **Technology** | `AAPL`, `MSFT`, `NVDA`, `AMD`, `INTC`, `ORCL`, `CSCO`, `QCOM` | 8 |
| **Financials** | `JPM`, `BAC`, `V`, `MA` | 4 |
| **Consumer** | `WMT`, `COST`, `KO`, `PG`, `HD`, `PEP` | 6 |
| **Healthcare** | `JNJ`, `PFE` | 2 |
| **Energy / Industry** | `XOM`, `CVX`, `CAT` | 3 |
| **Media / Entertainment** | `DIS`, `NFLX` | 2 |

We can safely use pandas ```read_csv``` even though files are ```.txt```, because they are comma separated.